# WebGL Viewer in Jupyter Notebooks

Pycortex can embed the interactive 3D brain viewer directly inside
Jupyter notebook cells. This page demonstrates the static viewer, which
generates a self-contained HTML directory with all the meshes and data
baked in.

There are two display methods:

| Method | How it works | Best for |
|--------|-------------|----------|
| **IFrame** (default) | Starts a Tornado server in a background thread | Live sessions with full interactivity (WebSocket control, surface morphing) |
| **Static** | Generates a viewer directory served by a lightweight HTTP server | Sharing notebooks, documentation, quick previews |

The static viewer is shown below because it works in pre-rendered
documentation. For the live IFrame mode, see the code examples at
the end of this page.

## Create example data

We start by creating a random volume mapped onto the example subject **S1**.

In [ ]:
import cortex
import numpy as np

np.random.seed(1234)
volume = cortex.Volume.random(subject="S1", xfmname="fullhead")
print(volume)

## Static viewer

Unlike most Jupyter widgets, the pycortex WebGL viewer **writes files to
disk**. `cortex.webgl.make_static` generates a complete viewer directory
containing HTML, JavaScript, CTM surface meshes, and JSON data files.
This is necessary because the 3D viewer relies on loading binary mesh
data and textures that cannot be embedded in a single notebook cell.

When using the convenience wrapper `cortex.webgl.jupyter.display(...,
method="static")`, temporary files are created automatically and cleaned
up when you call `viewer.close()` or use the context manager (`with`
statement). When using `make_static` directly (as below), the output
directory persists until you delete it yourself.

In [ ]:
from IPython.display import IFrame

cortex.webgl.make_static("static_viewer", volume, html_embed=True)
IFrame(src="static_viewer/index.html", width="100%", height="600px")

Try clicking and dragging the brain above to rotate it, or use the
scroll wheel to zoom. The toolbar on the right lets you toggle overlays,
adjust the colormap, and switch surface types.

## Using the Jupyter helper (live sessions)

For live Jupyter sessions, pycortex provides
`cortex.webgl.jupyter.display()`, which wraps the steps above into
a single call and manages server lifecycle automatically.

### IFrame mode (full interactivity)

```python
# Starts a Tornado server and embeds the viewer
server = cortex.webgl.jupyter.display(volume)
```

### Static mode with context manager

```python
# Generates a static viewer served via a local HTTP server.
# The context manager shuts down the server and cleans up temp files.
with cortex.webgl.jupyter.display(volume, method="static") as viewer:
    pass  # viewer is displayed above; do other work here
# Temp files and server are cleaned up automatically
```

### Multiple datasets

```python
# Pass a Dataset to display multiple data layers with a switcher
ds = cortex.Dataset(random1=volume, random2=cortex.Volume.random("S1", "fullhead"))
viewer = cortex.webgl.jupyter.display(ds, method="static")

# When done, clean up:
viewer.close()
```

### Programmatic control (IFrame mode only)

```python
server = cortex.webgl.jupyter.display(volume)

# In a subsequent cell, get a WebSocket control handle
client = server.get_client()

# Rotate the view
client._set_view(azimuth=45, altitude=30)

# Switch to inflated surface (mix=1.0 is fully inflated)
client._set_view(mix=1.0)

# Capture a screenshot
client.getImage("screenshot.png")
```

> **Note:** `get_client()` blocks until the browser connects via WebSocket,
> so always call it in a **separate cell** from `display()`.

## Customizing the viewer

Both methods accept the same keyword arguments as `cortex.webgl.show()`
and `cortex.webgl.make_static()`:

```python
cortex.webgl.jupyter.display(
    volume,
    types=("inflated",),           # surface types to include
    overlays_visible=("sulci",),   # show sulci overlay by default
    height=400,                     # shorter viewer
    title="My Experiment",
)
```

### Remote Jupyter setups

If you are running Jupyter on a remote server (JupyterHub, SSH tunnel,
cloud VM), set the appropriate environment variable so the IFrame URL
resolves correctly:

```bash
# For the IFrame/Tornado mode:
export CORTEX_JUPYTER_IFRAME_HOST=my-server.example.com

# For the static mode:
export CORTEX_JUPYTER_STATIC_HOST=0.0.0.0
```